In [1]:
import numpy as np
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
import time

# synthetic sanity test: 5 variables, 500 samples, pure random data
np.random.seed(0)
X_test = np.random.randn(500, 5)

start = time.time()
cg_test = pc(data=X_test, alpha=0.05, indep_test=fisherz, stable=True, verbose=False, show_progress=False)
print(f"Sanity test completed in {time.time()-start:.2f}s")
print("Nodes:", len(cg_test.G.nodes))

Sanity test completed in 0.00s
Nodes: 5


c:\Users\user\Desktop\ai causal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np
import json
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
X_pc_full_90 = np.load(os.path.join(out_dir, "pc_input_smoking_90pct.npy"))
with open(os.path.join(out_dir, "pc_col_names_smoking_90pct.json")) as f:
    col_names_90 = json.load(f)

print("Full data shape:", X_pc_full_90.shape)

# test with just the first 9 SNP columns + outcome, to match our previously-successful run size
X_small = X_pc_full_90[:, list(range(9)) + [-1]]
col_names_small = col_names_90[:9] + ["smoking_status"]

from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode
import time

n_nodes = len(col_names_small)
outcome_idx = col_names_small.index("smoking_status")
bk = BackgroundKnowledge()
nodes = [GraphNode(name) for name in col_names_small]
for i in range(n_nodes - 1):
    bk.add_node_to_tier(nodes[i], 0)
bk.add_node_to_tier(nodes[outcome_idx], 1)

start = time.time()
cg_small = pc(data=X_small, alpha=0.001, indep_test=fisherz, stable=True,
              uc_rule=0, uc_priority=2, background_knowledge=bk,
              verbose=False, show_progress=False, node_names=col_names_small)
print(f"9-SNP subset completed in {time.time()-start:.2f}s")

Full data shape: (3036, 68)
9-SNP subset completed in 0.16s


c:\Users\user\Desktop\ai causal\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\user\Desktop\ai causal\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [3]:
for n in [16, 18, 20, 22, 24]:
    X_sub = X_pc_full_90[:, list(range(n)) + [-1]]
    col_names_sub = col_names_90[:n] + ["smoking_status"]

    n_nodes_sub = len(col_names_sub)
    outcome_idx_sub = col_names_sub.index("smoking_status")
    bk_sub = BackgroundKnowledge()
    nodes_sub = [GraphNode(name) for name in col_names_sub]
    for i in range(n_nodes_sub - 1):
        bk_sub.add_node_to_tier(nodes_sub[i], 0)
    bk_sub.add_node_to_tier(nodes_sub[outcome_idx_sub], 1)

    print(f"Testing n={n}...")
    start = time.time()
    cg_sub = pc(data=X_sub, alpha=0.001, indep_test=fisherz, stable=True,
                uc_rule=0, uc_priority=2, background_knowledge=bk_sub,
                verbose=False, show_progress=False, node_names=col_names_sub)
    print(f"  n={n} completed in {time.time()-start:.2f}s")

Testing n=16...
  n=16 completed in 43.27s
Testing n=18...
  n=18 completed in 277.31s
Testing n=20...
  n=20 completed in 622.76s
Testing n=22...


KeyboardInterrupt: 

In [4]:
import numpy as np
from scipy import stats

X_snps_67 = X_pc_full_90[:, :-1]  # 67 SNPs, exclude smoking_status
n = X_snps_67.shape[0]

corr = np.corrcoef(X_snps_67.T)
np.fill_diagonal(corr, 0)

# convert correlation to p-value via Fisher Z (same test PC algorithm uses internally)
z = np.arctanh(np.clip(corr, -0.9999, 0.9999)) * np.sqrt(n - 3)
pvals = 2 * (1 - stats.norm.cdf(np.abs(z)))

alpha = 0.001
sig_mask = pvals < alpha
degree_per_snp = sig_mask.sum(axis=1)

print("Max marginal 'degree' (significant correlations) for any SNP:", degree_per_snp.max())
print("Mean degree:", degree_per_snp.mean())
print("\nTop 10 highest-degree SNPs (likely blowup culprits):")
top10_idx = np.argsort(-degree_per_snp)[:10]
for idx in top10_idx:
    print(f"  {col_names_90[idx]}: degree={degree_per_snp[idx]}")

Max marginal 'degree' (significant correlations) for any SNP: 12
Mean degree: 2.3582089552238807

Top 10 highest-degree SNPs (likely blowup culprits):
  exm523260-0_B_R_2058857401: degree=12
  exm822505-0_B_F_2058867002: degree=11
  exm601090-0_B_F_1918525937: degree=10
  exm598105-0_B_F_1918589067: degree=10
  exm1104413-0_T_F_1922787411: degree=10
  exm209101-0_B_F_1918911023: degree=9
  exm317601-0_B_R_1922250096: degree=9
  exm19206-0_T_F_1921571457: degree=9
  exm1148397-0_T_F_1922872045: degree=9
  exm683748-0_T_F_1922975213: degree=6


In [5]:
import numpy as np
import json
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

# remove the 5 highest-degree hub SNPs
hub_snps_to_remove = {col_names_90[idx] for idx in top10_idx[:5]}
print("Removing:", hub_snps_to_remove)

keep_indices = [i for i, name in enumerate(col_names_90[:-1]) if name not in hub_snps_to_remove]
X_pc_reduced = X_pc_full_90[:, keep_indices + [len(col_names_90)-1]]
col_names_reduced = [col_names_90[i] for i in keep_indices] + ["smoking_status"]

print(f"Reduced from {len(col_names_90)} to {len(col_names_reduced)} columns")

np.save(os.path.join(out_dir, "pc_input_smoking_90pct_reduced.npy"), X_pc_reduced)
with open(os.path.join(out_dir, "pc_col_names_smoking_90pct_reduced.json"), "w") as f:
    json.dump(col_names_reduced, f)

Removing: {'exm523260-0_B_R_2058857401', 'exm598105-0_B_F_1918589067', 'exm1104413-0_T_F_1922787411', 'exm822505-0_B_F_2058867002', 'exm601090-0_B_F_1918525937'}
Reduced from 68 to 63 columns


In [6]:
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode
import time

n_nodes = len(col_names_reduced)
outcome_idx = col_names_reduced.index("smoking_status")
bk = BackgroundKnowledge()
nodes = [GraphNode(name) for name in col_names_reduced]
for i in range(n_nodes - 1):
    bk.add_node_to_tier(nodes[i], 0)
bk.add_node_to_tier(nodes[outcome_idx], 1)

start = time.time()
cg = pc(data=X_pc_reduced, alpha=0.001, indep_test=fisherz, stable=True,
        uc_rule=0, uc_priority=2, background_knowledge=bk,
        depth=4,   # <-- caps conditioning set size, prevents runaway search
        verbose=False, show_progress=False, node_names=col_names_reduced)
print(f"Completed in {time.time()-start:.1f}s, nodes: {len(cg.G.nodes)}")

KeyboardInterrupt: 